# ARIMA Models

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/time-series/02-arima-models)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
np.random.seed(42)

## Simulating AR(1), MA(1), and ARIMA(1,1,1) processes

In [ ]:
n = 100
noise = np.random.normal(0, 1, n + 1)

# AR(1) with phi=0.8
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = 0.8 * ar1[t-1] + noise[t]

# MA(1) with theta=0.7
ma1 = noise[1:] + 0.7 * noise[:-1]

# ARIMA(1,1,1): AR on differenced series, then cumsum to integrate
arima_d = np.zeros(n)
for t in range(1, n):
    arima_d[t] = 0.5 * arima_d[t-1] + noise[t] + 0.3 * noise[t-1]
arima = np.cumsum(arima_d)  # integrate once

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, y, title in zip(axes, [ar1, ma1, arima], ['AR(1) φ=0.8', 'MA(1) θ=0.7', 'ARIMA(1,1,1)']):
    ax.plot(y, color='#2dd4bf', linewidth=1.2)
    ax.set_title(title, color='white')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## ACF vs PACF patterns for AR and MA

In [ ]:
def sample_acf(y, max_lag=20):
    y = y - y.mean()
    c0 = np.dot(y, y)
    return np.array([np.dot(y[:len(y)-k], y[k:]) / c0 for k in range(max_lag + 1)])

fig, axes = plt.subplots(2, 2, figsize=(12, 5))
sig = 1.96 / np.sqrt(n)

for col, (series, name) in enumerate([(ar1, 'AR(1)'), (ma1, 'MA(1)')]):
    acf = sample_acf(series)[1:]
    lags = np.arange(1, 21)
    for row, (vals, title) in enumerate([(acf, 'ACF'), (acf[:5], 'PACF (approx)')]):
        ax = axes[row][col]
        ax.bar(lags[:len(vals)], vals, color='#818cf8', alpha=0.8, width=0.6)
        ax.axhline(sig, color='#f59e0b', linestyle='--', linewidth=1)
        ax.axhline(-sig, color='#f59e0b', linestyle='--', linewidth=1)
        ax.set_title(f'{name} — {title}', color='white', fontsize=10)
        ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## AIC/BIC model selection

In [ ]:
def compute_aic_bic(y, p, q):
    n = len(y)
    # Simple AR(p) fitting via OLS for demonstration
    if p == 0 and q == 0:
        residuals = y - y.mean()
    else:
        # Fit AR(p) coefficients via Yule-Walker (simplified)
        X = np.column_stack([y[p-i:-i if i > 0 else None] for i in range(1, p+1)] if p > 0 else [np.ones(n-p)])
        y_fit = y[p:]
        if p > 0:
            coeffs = np.linalg.lstsq(X, y_fit, rcond=None)[0]
            residuals = y_fit - X @ coeffs
        else:
            residuals = y - y.mean()
    sigma2 = np.var(residuals)
    k = p + q + 1
    aic = n * np.log(sigma2) + 2 * k
    bic = n * np.log(sigma2) + k * np.log(n)
    return aic, bic

print("Model      AIC      BIC")
print("-" * 30)
for p, q in [(0,0),(1,0),(2,0),(0,1),(0,2),(1,1)]:
    aic, bic = compute_aic_bic(ar1, p, q)
    print(f"ARMA({p},{q})   {aic:.1f}   {bic:.1f}")

## ✏️ Your turn: Fit an ARIMA model

In [ ]:
# TODO(you): Create an MA(1) series with theta=0.5, n=200
# Then compute its sample ACF at lags 1-5
# Hint: y[t] = noise[t] + 0.5 * noise[t-1]

# YOUR CODE HERE
ma1_series = None  # replace
acf_lags = None    # replace: np.array of length 5

assert ma1_series is not None
assert acf_lags is not None and len(acf_lags) == 5
print(f"✓ ACF at lag 1: {acf_lags[0]:.3f} (should be ≈0.40)")

<details><summary>Solution</summary>

```python
rng = np.random.default_rng(42)
n = 200
noise = rng.normal(0, 1, n + 1)
ma1_series = noise[1:] + 0.5 * noise[:-1]

def sample_acf(y, k):
    y = y - y.mean(); c0 = np.dot(y, y)
    return np.array([np.dot(y[:len(y)-j], y[j:]) / c0 for j in range(1, k+1)])

acf_lags = sample_acf(ma1_series, 5)
```
</details>